# 18 — Client Watchlist Report

Báo cáo chi tiết cho khách hàng về danh mục khuyến nghị và kịch bản phân tích.

In [2]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [3]:
# Cell: Import Report Modules
import pandas as pd

from src.analytics.entropy import build_market_entropy_profile
from src.analytics.correlation import build_return_matrix, compute_correlation_matrix
from src.data.loaders import read_parquet
from src.reporting.chart_factory import (
    correlation_heatmap,
    entropy_regime_chart,
    monte_carlo_exit_timeline_chart,
    monte_carlo_scenario_chart,
    monte_carlo_terminal_distribution_chart,
    risk_contribution_chart,
    sharpe_stability_chart,
    weight_comparison_chart,
)
from src.risk.management import (
    build_portfolio_risk_report,
    expanding_sharpe_series,
    portfolio_return_series,
    rolling_sharpe_series,
)

In [4]:
# Cell: Load Report Inputs
market_df = read_parquet("data/processed/market_features.parquet")
trade_cards = read_parquet("data/signals/breakout_20d_trade_cards.parquet")
meta_ranking = read_parquet("data/signals/meta_ranking.parquet")
stealth_watchlist = read_parquet("data/signals/stealth_accumulation_watchlist.parquet")
stealth_sector_watchlist = read_parquet("data/signals/stealth_accumulation_sector_watchlist.parquet")
setup_signals = read_parquet("data/signals/setup_signals.parquet")
capital_plan = read_parquet("data/signals/capital_allocation_plan.parquet")
weights_overlay = read_parquet("data/signals/risk_overlay_weights.parquet")
benchmark_comparison = read_parquet("data/signals/benchmark_comparison.parquet")
mc_scenarios = read_parquet("data/signals/monte_carlo_scenarios.parquet")
mc_exits = read_parquet("data/signals/monte_carlo_exits.parquet")
mc_path_summary = read_parquet("data/signals/monte_carlo_path_summary.parquet")

for df_name in ["market_df", "trade_cards", "meta_ranking", "stealth_watchlist", "stealth_sector_watchlist", "setup_signals", "capital_plan"]:
    df = globals()[df_name]
    if not df.empty and "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])

summary = {
    "market_rows": len(market_df),
    "trade_cards_rows": len(trade_cards),
    "meta_ranking_rows": len(meta_ranking),
    "stealth_watchlist_rows": len(stealth_watchlist),
    "capital_plan_rows": len(capital_plan),
}
pd.DataFrame([summary]).T.rename(columns={0: "value"})


,value
market_rows,1617
trade_cards_rows,499
meta_ranking_rows,499
stealth_watchlist_rows,3
capital_plan_rows,4


In [5]:
# Cell: Build Client Watchlist Table
# CUSTOMIZABLE: Define your custom watchlist here to filter client watchlist report, or leave as None
user_watchlist = None  # e.g., ["SSI", "HPG", "FPT", "VIC", "VNM"]
watch_actions = [
    "Watchlist - No Signal",
    "Watchlist - Emerging",
    "Watchlist - Conditional",
    "Valid Breakout Setup",
    "Watch - Limited Sample Breakout",
    "Watch - Conditional Breakout",
    "Watch - Need Confirmation",
]
watchlist = trade_cards[trade_cards.get("action", pd.Series(dtype=object)).isin(watch_actions)].copy()
if user_watchlist is not None:
    watchlist_upper = [sym.upper() for sym in user_watchlist]
    # Keep/rebuild trade card data for those symbols
    watchlist = trade_cards[trade_cards["symbol"].str.upper().isin(watchlist_upper)].copy()
    if watchlist.empty:
        # construct mock entries for manual watchlist tickers if not in trade_cards
        watchlist = pd.DataFrame([{"symbol": sym, "action": "Watchlist - Manual", "edge_tier": "Valid Edge"} for sym in watchlist_upper])
    print(f"Using user manual report watchlist: {watchlist_upper}")
if not meta_ranking.empty and "symbol" in meta_ranking.columns:
    enrich_cols = [col for col in ["symbol", "meta_score", "priority_bucket", "market_regime", "watchlist_flag"] if col in meta_ranking.columns]
    watchlist = watchlist.merge(meta_ranking[enrich_cols].drop_duplicates(subset=["symbol"]), on="symbol", how="left")
# Fallback values
if "meta_score" in watchlist.columns:
    watchlist["meta_score"] = watchlist["meta_score"].fillna(50)
if "priority_bucket" in watchlist.columns:
    watchlist["priority_bucket"] = watchlist["priority_bucket"].fillna("Medium Priority")
sort_cols = [col for col in ["trade_edge_score", "setup_quality_score", "meta_score", "profit_factor"] if col in watchlist.columns]
watchlist = watchlist.sort_values(sort_cols, ascending=False).reset_index(drop=True) if sort_cols else watchlist
report_cols = [
    "symbol",
    "date",
    "close",
    "action",
    "edge_tier",
    "volatility_regime",
    "volatility_rule",
    "setup_quality_score",
    "trade_edge_score",
    "meta_score",
    "priority_bucket",
]
display(watchlist[[col for col in report_cols if col in watchlist.columns]].head(15))

,symbol,date,close,action,edge_tier,volatility_regime,volatility_rule,setup_quality_score,trade_edge_score,meta_score,priority_bucket
0,ASP,2026-06-29,8910,Watch - Conditional Breakout,Conditional Edge,Stress Vol,Allowed,75,75,57.959447,Secondary Watchlist
1,VCG,2026-06-29,20950,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,76,0,42.308108,Low Priority
2,CTS,2026-06-29,25100,Watchlist - No Signal,Valid Edge,High Vol,Allowed,74,0,38.272331,Low Priority
3,BID,2026-06-29,42350,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,61.131703,High Conviction Watchlist
4,STB,2026-06-29,75800,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,43.840885,Low Priority
5,MSB,2026-06-29,16000,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,42.936548,Low Priority
6,VAB,2026-06-29,11250,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,42.600549,Low Priority
7,OCB,2026-06-29,12450,Watchlist - Emerging,Emerging Edge,High Vol,Allowed,72,0,42.326235,Low Priority
8,VCB,2026-06-29,62100,Watchlist - Emerging,Emerging Edge,Low Vol,Allowed,72,0,41.348560,Low Priority
9,VIC,2026-06-29,218300,Watchlist - Emerging,Emerging Edge,Low Vol,Allowed,72,0,40.445807,Low Priority


In [6]:
# Cell: Market Context Section
if market_df.empty:
    print("Khong co market_df. Hay chay notebook 06 truoc.")
else:
    if "market_entropy_21d" not in market_df.columns and not setup_signals.empty:
        market_df = build_market_entropy_profile(setup_signals)
    latest_market = market_df.sort_values("date").tail(1)
    display(latest_market[[col for col in ["date", "market_regime", "market_entropy_21d", "market_entropy_63d", "market_entropy_regime", "market_entropy_shift", "advance_ratio"] if col in latest_market.columns]])
    display(entropy_regime_chart(market_df.sort_values("date"), entropy_col="market_entropy_21d", regime_col="market_entropy_regime", title="Client Report - VN Market Context"))

,date,market_regime,market_entropy_21d,market_entropy_63d,market_entropy_regime,market_entropy_shift,advance_ratio
1616,2026-06-29,Risk-off,0.80474,0.704219,Chaotic,Stable,0.490826


## Client Text - Market Context

Thi truong hien tai duoc danh gia thong qua market regime, entropy va do rong thi truong. Khi entropy o muc thap - trung binh va volatility regime khong qua bat loi, xac suat xuat hien cac co hoi breakout/chat luong watchlist thuong tot hon mat bang chung.

In [8]:
# Cell: Watchlist Summary Section
watchlist_no_signal = watchlist[watchlist["action"] == "Watchlist - No Signal"].copy() if "action" in watchlist.columns else pd.DataFrame()
watchlist_conditional = watchlist[watchlist["action"] == "Watchlist - Conditional"].copy() if "action" in watchlist.columns else pd.DataFrame()
actionable = watchlist[watchlist["action"].isin(["Valid Breakout Setup", "Watch - Conditional Breakout"])].copy() if "action" in watchlist.columns else pd.DataFrame()

summary = {
    "watchlist_total": len(watchlist),
    "watchlist_no_signal": len(watchlist_no_signal),
    "watchlist_conditional": len(watchlist_conditional),
    "actionable_setups": len(actionable),
}
display(pd.DataFrame([summary]))
display(watchlist_no_signal.head(10))
display(watchlist_conditional.head(10))

,watchlist_total,watchlist_no_signal,watchlist_conditional,actionable_setups
0,81,7,26,1


,symbol,date,close,breakout_20d,edge_tier,eligible,watchlist_eligible,volatility_regime,volatility_rule,profit_factor,...,stop_loss,holding_period,setup_quality_score,trade_edge_score,action,data_is_saying,meta_score,priority_bucket,market_regime,watchlist_flag
2,CTS,2026-06-29,25100,False,Valid Edge,True,True,High Vol,Allowed,1.709146,...,-0.05,20.0,74,0,Watchlist - No Signal,"CTS: chưa có breakout 20D mới, nhưng mã này nằ...",38.272331,Low Priority,None,False
14,HDB,2026-06-29,25750,False,Valid Edge,True,True,Low Vol,Allowed,60.437820,...,-0.05,20.0,67,0,Watchlist - No Signal,"HDB: chưa có breakout 20D mới, nhưng mã này nằ...",38.174540,Low Priority,None,False
15,CTG,2026-06-29,33950,False,Valid Edge,True,True,Low Vol,Allowed,1.569465,...,-0.05,20.0,67,0,Watchlist - No Signal,"CTG: chưa có breakout 20D mới, nhưng mã này nằ...",36.774715,Low Priority,None,False
22,NAF,2026-06-29,49950,False,Valid Edge,True,True,Low Vol,Allowed,3.743964,...,-0.05,20.0,66,0,Watchlist - No Signal,"NAF: chưa có breakout 20D mới, nhưng mã này nằ...",32.918747,Low Priority,None,False
23,BFC,2026-06-29,56100,False,Valid Edge,True,True,Low Vol,Allowed,1.526095,...,-0.05,20.0,66,0,Watchlist - No Signal,"BFC: chưa có breakout 20D mới, nhưng mã này nằ...",29.068355,Low Priority,None,False
30,SZC,2026-06-29,22350,False,Valid Edge,True,True,Low Vol,Allowed,23.162268,...,-0.05,20.0,60,0,Watchlist - No Signal,"SZC: chưa có breakout 20D mới, nhưng mã này nằ...",27.024056,Low Priority,None,False
31,DIG,2026-06-29,12350,False,Valid Edge,True,True,Low Vol,Allowed,2.408283,...,-0.05,20.0,60,0,Watchlist - No Signal,"DIG: chưa có breakout 20D mới, nhưng mã này nằ...",24.655555,Low Priority,None,False


,symbol,date,close,breakout_20d,edge_tier,eligible,watchlist_eligible,volatility_regime,volatility_rule,profit_factor,...,stop_loss,holding_period,setup_quality_score,trade_edge_score,action,data_is_saying,meta_score,priority_bucket,market_regime,watchlist_flag
12,VIB,2026-06-29,16500,False,Conditional Edge,False,True,Normal Vol,Allowed,1.303627,...,-0.05,20.0,69,0,Watchlist - Conditional,VIB: chưa có breakout 20D mới. Mã có edge ở mứ...,40.177569,Low Priority,None,False
24,KLB,2026-06-29,16400,False,Conditional Edge,False,True,Normal Vol,Allowed,2.360321,...,-0.05,20.0,65,0,Watchlist - Conditional,KLB: chưa có breakout 20D mới. Mã có edge ở mứ...,42.220109,Low Priority,None,False
25,TNT,2026-06-24,12150,False,Conditional Edge,False,True,Normal Vol,Allowed,1.760245,...,-0.05,20.0,65,0,Watchlist - Conditional,TNT: chưa có breakout 20D mới. Mã có edge ở mứ...,40.332794,Low Priority,None,False
26,ABB,2026-06-29,18000,False,Conditional Edge,False,True,Normal Vol,Allowed,2.964336,...,-0.05,20.0,65,0,Watchlist - Conditional,ABB: chưa có breakout 20D mới. Mã có edge ở mứ...,38.341949,Low Priority,None,False
27,SSB,2026-06-29,16100,False,Conditional Edge,False,True,High Vol,Allowed,1.381543,...,-0.05,20.0,65,0,Watchlist - Conditional,SSB: chưa có breakout 20D mới. Mã có edge ở mứ...,38.142156,Low Priority,None,False
32,VPB,2026-06-29,27100,False,Conditional Edge,False,True,Low Vol,Allowed,1.373959,...,-0.05,20.0,59,0,Watchlist - Conditional,VPB: chưa có breakout 20D mới. Mã có edge ở mứ...,38.056280,Low Priority,None,False
33,ORS,2026-06-29,13450,False,Conditional Edge,False,True,Normal Vol,Allowed,2.117065,...,-0.05,20.0,59,0,Watchlist - Conditional,ORS: chưa có breakout 20D mới. Mã có edge ở mứ...,38.048360,Low Priority,None,False
34,FTS,2026-06-29,26500,False,Conditional Edge,False,True,High Vol,Allowed,3.770287,...,-0.05,20.0,59,0,Watchlist - Conditional,FTS: chưa có breakout 20D mới. Mã có edge ở mứ...,36.597291,Low Priority,None,False
36,DCL,2026-06-29,37800,False,Conditional Edge,False,True,Low Vol,Allowed,1.821266,...,-0.05,20.0,59,0,Watchlist - Conditional,DCL: chưa có breakout 20D mới. Mã có edge ở mứ...,36.111828,Low Priority,None,False
41,MBB,2026-06-29,25200,False,Conditional Edge,False,True,Low Vol,Allowed,2.059835,...,-0.05,20.0,58,0,Watchlist - Conditional,MBB: chưa có breakout 20D mới. Mã có edge ở mứ...,38.277194,Low Priority,None,False


## Client Text - Why These Names

Danh sach nay khong duoc chon vi tang nong ngan han, ma vi he thong loc ra cac ma co edge lich su tot tren chinh setup dang theo doi, chat luong setup hien tai cao, va boi canh bien dong chua gay bat loi lon cho chien luoc. Nhom `Watchlist - No Signal` la cac ma dang duoc uu tien theo doi nhung chua xuat hien trigger vao lenh. Nhom `Watchlist - Conditional` la cac ma co lua chon giao dich tiem nang, nhung can them mot dieu kien xac nhan nua truoc khi hanh dong.

In [10]:
# Cell: Meta Ranking Section
if meta_ranking.empty:
    print("Khong co meta ranking output. Hay chay notebook 16 truoc.")
else:
    display(meta_ranking[[col for col in ["symbol", "meta_score", "trade_edge_score", "setup_quality_score", "net_accumulation_score", "priority_bucket", "market_regime", "volatility_regime"] if col in meta_ranking.columns]].head(12))

,symbol,meta_score,trade_edge_score,setup_quality_score,net_accumulation_score,priority_bucket,market_regime,volatility_regime
0,BID,61.131703,0,72,67.171062,High Conviction Watchlist,Unknown,Normal Vol
1,SAB,59.710793,0,50,71.315483,Secondary Watchlist,Unknown,Low Vol
2,ASP,57.959447,75,75,0.000000,Secondary Watchlist,None,Stress Vol
3,VPI,53.730420,0,50,65.759453,Secondary Watchlist,Unknown,Low Vol
4,STB,43.840885,0,72,0.000000,Low Priority,None,Normal Vol
5,MSB,42.936548,0,72,0.000000,Low Priority,None,Normal Vol
6,VAB,42.600549,0,72,0.000000,Low Priority,None,Normal Vol
7,OCB,42.326235,0,72,0.000000,Low Priority,None,High Vol
8,VCG,42.308108,0,76,0.000000,Low Priority,None,Normal Vol
9,KLB,42.220109,0,65,0.000000,Low Priority,None,Normal Vol


In [11]:
# Cell: Correlation And Diversification Section
if watchlist.empty or setup_signals.empty:
    print("Khong du du lieu de build correlation section.")
else:
    symbols = watchlist["symbol"].dropna().astype(str).unique().tolist()[:12]
    history = setup_signals[setup_signals["symbol"].astype(str).isin(symbols)].copy()
    if not history.empty:
        return_matrix = build_return_matrix(history, return_col="return_1d").dropna(how="all")
        corr = compute_correlation_matrix(return_matrix, min_periods=20)
        display(corr)
        display(correlation_heatmap(corr, title="Client Report - Watchlist Correlation"))

symbol,ASP,BID,CTS,HHP,MSB,OCB,PVP,STB,VAB,VCB,VCG,VIC
symbol,,,,,,,,,,,,
ASP,1.000000,0.121455,0.050346,0.033415,0.140170,0.139991,0.134013,0.095337,0.124857,0.052002,0.122539,-0.035141
BID,0.121455,1.000000,0.273920,0.212832,0.766614,0.766173,0.679305,0.223269,0.311213,0.854742,0.709102,-0.487244
CTS,0.050346,0.273920,1.000000,0.089474,0.290678,0.272993,0.232451,0.306272,0.333054,0.181201,0.377856,0.081931
HHP,0.033415,0.212832,0.089474,1.000000,0.115559,0.090128,-0.014971,0.042744,0.055667,0.191922,0.071406,0.141895
MSB,0.140170,0.766614,0.290678,0.115559,1.000000,0.877950,0.701870,0.064840,0.337215,0.770859,0.732182,-0.563720
OCB,0.139991,0.766173,0.272993,0.090128,0.877950,1.000000,0.683510,0.168412,0.356611,0.775450,0.741958,-0.531243
PVP,0.134013,0.679305,0.232451,-0.014971,0.701870,0.683510,1.000000,0.069461,-0.021237,0.677062,0.655758,-0.505566
STB,0.095337,0.223269,0.306272,0.042744,0.064840,0.168412,0.069461,1.000000,0.310633,0.031259,0.215497,0.196398
VAB,0.124857,0.311213,0.333054,0.055667,0.337215,0.356611,-0.021237,0.310633,1.000000,0.297397,0.277571,0.016556


## Client Text - Diversification Note

Khach hang khong nen chi nhin vao tung ma rieng le. Ma duoc dua vao watchlist van can xem xet theo cum nganh va muc do tuong quan, tranh tinh trang nhieu ma nhin co ve khac nhau nhung thuc chat dang cung chiu mot rui ro giong nhau.

In [13]:
# Cell: Monte Carlo Section
if mc_scenarios.empty and mc_exits.empty and mc_path_summary.empty:
    print("Khong co Monte Carlo outputs luu san. Hay chay notebook 08 va save outputs neu muon dung section nay.")
else:
    if not mc_scenarios.empty:
        display(mc_scenarios)
        display(monte_carlo_scenario_chart(mc_scenarios, title="Client Report - Monte Carlo Scenario Map"))
    if not mc_exits.empty:
        display(monte_carlo_exit_timeline_chart(mc_exits, title="Client Report - Exit Timing"))
    if not mc_path_summary.empty:
        cols = [col for col in ["day", "mean_price", "q05", "q50", "q95"] if col in mc_path_summary.columns]
        display(mc_path_summary[cols].head())

,day,mean_price,q05,q50,q95
0,0,23400.000000,23400.000000,23400.000000,23400.000000
1,1,23458.682624,22576.884422,23400.000000,24410.975272
2,2,23517.881501,21382.296691,23404.870527,25179.346891
3,3,23574.611320,18516.786435,23423.328668,27539.664031
4,4,23637.097999,17438.658793,23439.646035,31281.593246


## Client Text - Scenario Analysis

Monte Carlo khong du doan chac chan gia se di theo mot huong co dinh. Muc tieu cua phan nay la mo ta bien do ket qua co the xay ra, giup khach hang hinh dung truoc upside, downside va toc do xuat hien rui ro trong truong hop tham gia giao dich.

In [15]:
# Cell: Portfolio And Risk Section
if capital_plan.empty or setup_signals.empty:
    print("Khong co capital plan hoac setup_signals de build risk section.")
else:
    symbols = capital_plan["symbol"].dropna().astype(str).tolist()
    history = setup_signals[setup_signals["symbol"].astype(str).isin(symbols)].copy()
    if not history.empty:
        return_matrix = build_return_matrix(history, return_col="return_1d").dropna(how="all").fillna(0.0)
        if not weights_overlay.empty:
            weight_col = "target_weight" if "target_weight" in weights_overlay.columns else "weight"
            if "symbol" in weights_overlay.columns and weight_col in weights_overlay.columns:
                overlay_weights = weights_overlay.set_index("symbol")[weight_col]
            else:
                overlay_weights = pd.Series(dtype=float)
        else:
            overlay_weights = capital_plan.set_index("symbol").get("gross_weight", pd.Series(dtype=float))
        overlay_weights = overlay_weights[overlay_weights.index.isin(return_matrix.columns)]
        if not overlay_weights.empty:
            port_returns = portfolio_return_series(return_matrix, overlay_weights)
            risk_report = build_portfolio_risk_report(return_matrix, overlay_weights)
            exp_sharpe = expanding_sharpe_series(port_returns, min_periods=21)
            roll_sharpe = rolling_sharpe_series(port_returns, window=min(63, max(21, len(port_returns) // 3)))
            display(pd.DataFrame([risk_report]).T.rename(columns={0: "value"}))
            display(sharpe_stability_chart(exp_sharpe, roll_sharpe, title="Client Report - Sharpe Stability"))


,value
annual_return,0.094005
annual_volatility,0.118962
sharpe_ratio,0.814683
sortino_ratio,1.275787
max_drawdown,-0.107148
calmar_ratio,0.877336
var_95,-0.010826
cvar_95,-0.017313
expanding_sharpe_latest,0.814683
rolling_sharpe_latest,4.829212


In [16]:
# Cell: Capital Plan Section
if capital_plan.empty:
    print("Khong co capital plan output. Hay chay notebook 13 hoac 17 truoc.")
else:
    display(capital_plan[[col for col in ["capital_rank", "symbol", "edge_tier", "conviction_score_resolved", "volatility_regime", "target_weight", "gross_weight", "position_value", "risk_pct_effective"] if col in capital_plan.columns]])
    if {"symbol", "target_weight", "gross_weight"}.issubset(capital_plan.columns):
        display(weight_comparison_chart(capital_plan, title="Client Report - Target Vs Final Weights"))
    if "risk_contribution" in capital_plan.columns:
        display(risk_contribution_chart(capital_plan.set_index("symbol")["risk_contribution"], title="Client Report - Risk Contributions"))

,capital_rank,symbol,edge_tier,conviction_score_resolved,volatility_regime,target_weight,gross_weight,position_value,risk_pct_effective
0,1,ASP,Conditional Edge,75.0,Stress Vol,0.383577,0.164826,83112480.0,0.006250
1,2,VPI,Insufficient Data,39.5,Low Vol,0.383577,0.396604,199984800.0,0.008950
2,3,BID,Emerging Edge,74.0,Normal Vol,0.121677,0.229201,115573150.0,0.014229
3,4,SAB,Insufficient Data,57.5,Low Vol,0.111169,0.209368,105572550.0,0.014512


## Client Text - Action Plan

De xuat cuoi cung khong phai la mua toan bo danh sach. Watchlist nen duoc chia thanh 3 lop: nhom theo doi sat, nhom dieu kien, va nhom co the hanh dong neu xuat hien trigger. Cach tiep can nay giup khach hang khong mua duoi gia, dong thoi van giu duoc su chu dong khi co co hoi xac nhan ro rang.

In [18]:
# Cell: Final Client Summary Table
final_cols = [
    "symbol",
    "action",
    "edge_tier",
    "volatility_regime",
    "volatility_rule",
    "setup_quality_score",
    "trade_edge_score",
    "meta_score",
]
final_table = watchlist[[col for col in final_cols if col in watchlist.columns]].head(12).copy()
final_table

,symbol,action,edge_tier,volatility_regime,volatility_rule,setup_quality_score,trade_edge_score,meta_score
0,ASP,Watch - Conditional Breakout,Conditional Edge,Stress Vol,Allowed,75,75,57.959447
1,VCG,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,76,0,42.308108
2,CTS,Watchlist - No Signal,Valid Edge,High Vol,Allowed,74,0,38.272331
3,BID,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,61.131703
4,STB,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,43.840885
5,MSB,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,42.936548
6,VAB,Watchlist - Emerging,Emerging Edge,Normal Vol,Allowed,72,0,42.600549
7,OCB,Watchlist - Emerging,Emerging Edge,High Vol,Allowed,72,0,42.326235
8,VCB,Watchlist - Emerging,Emerging Edge,Low Vol,Allowed,72,0,41.348560
9,VIC,Watchlist - Emerging,Emerging Edge,Low Vol,Allowed,72,0,40.445807
